# Network Flow V2 — ToN-IoT Network: Step 02 / Feature and Data Quality

**Input:** `ton_iot_train_test_network.csv` in `data/raw/ton_iot/`. Run **Step 01** for this dataset first.

This is a **read-only, chunk-based** audit. It checks binary/type labels, duplicate row signatures, conflicting labels for identical feature patterns, numeric parsing, zero-heavy/constant fields, and risky identifier/payload columns. It writes an aggregate JSON and draft feature dictionary.

**Memory:** fingerprints are retained as compact 64-bit hashes rather than collecting the entire CSV in a DataFrame. Fingerprint results are screening evidence (extremely unlikely hash collisions), not cryptographic proof.

**Important:** The common network split generally supplies binary label and multiclass type. This notebook verifies rather than assumes their presence or semantics. We do **not** automatically drop zeros, missing values, rare classes, or protocol-specific columns.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
from IPython.display import display
import hashlib
import json
import numpy as np
import pandas as pd

ROOT = Path(r"D:\J26-CS-336-Network-Flow-V2\network_flow_v2")
DATA = ROOT / "data" / "raw" / "ton_iot" / 'ton_iot_train_test_network.csv'
AUDIT_PATH = ROOT / "reports" / "audits" / "ton_iot_audit_v1.json"
REPORT_PATH = ROOT / "reports" / "audits" / "ton_iot_feature_quality_v1.json"
DOC_PATH = ROOT / "docs" / "TON_IOT_FEATURE_DICTIONARY_DRAFT.md"
CHUNK_SIZE = 25000
DISTINCT_CAP = 10000

if not AUDIT_PATH.is_file():
    raise FileNotFoundError("Run Step 01 first: " + str(AUDIT_PATH))
if not DATA.is_file():
    raise FileNotFoundError("Dataset not found: " + str(DATA))
with AUDIT_PATH.open("r", encoding="utf-8") as f:
    audit = json.load(f)

LABEL_COLUMNS = audit["candidate_label_columns"]
COLUMNS = audit["column_names"]
FEATURES = [c for c in COLUMNS if c not in LABEL_COLUMNS]
print("Dataset:", DATA.name)
print("Columns:", len(COLUMNS), "| candidate labels:", LABEL_COLUMNS)
print("Features inspected:", len(FEATURES))
print("Label meanings still require source-documentation verification.")


## 1. Confirm Step 01 checksum and determine provisional label levels

Do not use any label column in `X`. A `label` or `class1` value of 0/1 is a **candidate** normal/attack label, not automatically proven.

In [ ]:
digest = hashlib.sha256()
with DATA.open("rb") as f:
    for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
        digest.update(block)
if digest.hexdigest() != audit["sha256"]:
    raise ValueError("CSV changed since Step 01 — rerun Step 01 before Step 02")

PREFERRED_BINARY = ['label', 'Attack_label']
PREFERRED_CATEGORY = ['type', 'Attack_type']
binary_col = next((c for c in PREFERRED_BINARY if c in LABEL_COLUMNS), None)
category_col = next((c for c in PREFERRED_CATEGORY if c in LABEL_COLUMNS and c != binary_col), None)
print("Provisional binary label:", binary_col)
print("Provisional attack-category label:", category_col)
print("Confirm category semantics in the dataset documentation before training.")


## 2. Full feature quality & duplicate screen

Values are read as **text** consistently across chunks to avoid inconsistent type-based hashing. Zero frequency means numeric-zero-looking values divided by all rows; it does **not** prove absence of a protocol. Exact-value cardinalities are capped at 10,000 for memory safety.

In [ ]:
MISSING_TOKENS = {"", "?", "na", "n/a", "nan", "none", "null", "undefined", "-", "inf", "-inf"}
missing = Counter()
placeholders = Counter()
zeros = Counter()
parseable = Counter()
nonnull = Counter()
distinct = {c: set() for c in FEATURES}
distinct_capped = set()
full_hash_chunks = []
feature_hash_chunks = []
label_hash_chunks = {c: [] for c in LABEL_COLUMNS}
row_count = 0

for chunk_number, part in enumerate(pd.read_csv(DATA, chunksize=CHUNK_SIZE,
                                              dtype="string", low_memory=False), start=1):
    if part.columns.tolist() != COLUMNS:
        raise ValueError("Column names differ from Step 01 audit")
    n = len(part)
    row_count += n
    # Stable 64-bit fingerprints: same raw string cells, same column order, same hash.
    full_hash_chunks.append(pd.util.hash_pandas_object(
        part[COLUMNS].fillna("<PANDAS_NA>"), index=False).to_numpy(dtype=np.uint64))
    feature_hash_chunks.append(pd.util.hash_pandas_object(
        part[FEATURES].fillna("<PANDAS_NA>"), index=False).to_numpy(dtype=np.uint64))
    for label in LABEL_COLUMNS:
        label_hash_chunks[label].append(pd.util.hash_pandas_object(
            part[[label]].fillna("<PANDAS_NA>"), index=False).to_numpy(dtype=np.uint64))

    for col in FEATURES:
        s = part[col]
        missing[col] += int(s.isna().sum())
        nonnull[col] += int(s.notna().sum())
        norm = s.str.strip()
        placeholders[col] += int(norm.str.casefold().isin(MISSING_TOKENS).sum())
        numeric = pd.to_numeric(norm, errors="coerce")
        parseable[col] += int(numeric.notna().sum())
        zeros[col] += int((numeric == 0).sum())
        if col not in distinct_capped:
            # Capped but accurate below the cap, even if early unique values repeat.
            for value in norm.dropna().unique():
                distinct[col].add(value)
                if len(distinct[col]) > DISTINCT_CAP:
                    distinct_capped.add(col)
                    distinct[col].clear()
                    break
    if chunk_number % 10 == 0:
        print(f"Processed {row_count:,} rows ({chunk_number} chunks)")

print(f"Full feature-quality scan complete: {row_count:,} rows")
if row_count != int(audit["row_count"]):
    raise ValueError("Row count differs from Step 01")


## 3. Duplicate records and conflicting label groups

A **duplicate** is a repeated row or repeated candidate-input combination. A **conflict** means the same candidate-input fingerprint appears with different known labels. These checks help design leakage-aware splits; they do not establish capture independence.

In [ ]:
full_hash = np.concatenate(full_hash_chunks)
feature_hash = np.concatenate(feature_hash_chunks)
_, full_count = np.unique(full_hash, return_counts=True)
_, feature_count = np.unique(feature_hash, return_counts=True)

exact_rows_in_duplicate_groups = int(full_count[full_count > 1].sum())
excess_full_duplicates = int((full_count[full_count > 1] - 1).sum())
feature_rows_in_duplicate_groups = int(feature_count[feature_count > 1].sum())
excess_feature_duplicates = int((feature_count[feature_count > 1] - 1).sum())

print("Rows in identical full-record groups:", exact_rows_in_duplicate_groups)
print("Excess identical rows:", excess_full_duplicates)
print("Rows in identical candidate-feature groups:", feature_rows_in_duplicate_groups)
print("Excess candidate-feature duplicates:", excess_feature_duplicates)

label_conflicts = {}
for label in LABEL_COLUMNS:
    hashes = np.concatenate(label_hash_chunks[label])
    ordered = np.lexsort((hashes, feature_hash))
    sorted_fp = feature_hash[ordered]
    sorted_label = hashes[ordered]
    conflicting_boundary = (sorted_fp[1:] == sorted_fp[:-1]) & (sorted_label[1:] != sorted_label[:-1])
    conflicting_groups = np.unique(sorted_fp[:-1][conflicting_boundary])
    label_conflicts[label] = int(len(conflicting_groups))
    print(f"Feature groups with differing {label!r} labels:", label_conflicts[label])

del full_hash_chunks, feature_hash_chunks, label_hash_chunks


## 4. Zero-heavy, constant, mixed-data and possible leakage fields

This prepares a review table. **Do not automatically remove zero-heavy fields**; IoT traffic naturally contains protocol-specific zeros. Constant features have no variance in this particular CSV, but feature removal for a deployed classifier should be validated against future traffic and training splits.

In [ ]:
quality_rows=[]
for col in FEATURES:
    is_capped = col in distinct_capped
    q={
        "feature":col,
        "missing_count":int(missing[col]),
        "placeholder_token_count":int(placeholders[col]),
        "numeric_parse_pct":round(100 * parseable[col] / max(nonnull[col],1), 2),
        "numeric_zero_pct_of_all_rows":round(100 * zeros[col] / max(row_count,1), 2),
        "distinct_values":None if is_capped else int(len(distinct[col])),
        "distinct_count_at_least":DISTINCT_CAP + 1 if is_capped else int(len(distinct[col])),
        "distinct_capped":is_capped,
        "review_as_identifier_or_payload":any(token in col.casefold() for token in (
            "ip.", "src_ip", "dst_ip", "addr", "id", "time", "timestamp", "payload", "content", "uri", "msg", "session")),
    }
    quality_rows.append(q)
quality = pd.DataFrame(quality_rows)
print("Potential constant non-label fields:")
display(quality[(quality.distinct_values == 1)][["feature", "distinct_values"]])
print("Features with >=95% numeric zero values (investigate):")
display(quality[quality.numeric_zero_pct_of_all_rows >= 95]
        [["feature", "numeric_zero_pct_of_all_rows", "distinct_count_at_least"]]
        .sort_values("numeric_zero_pct_of_all_rows", ascending=False).head(35))
print("Missing/placeholder markers:")
display(quality.sort_values(["missing_count","placeholder_token_count"], ascending=False)
        [["feature", "missing_count", "placeholder_token_count"]].head(25))
print("Possible identifiers, time, payload or text features needing leakage/privacy review:")
display(quality[quality.review_as_identifier_or_payload][["feature", "numeric_parse_pct", "distinct_count_at_least"]].head(45))


## 5. Check **binary versus category** label consistency (if semantics are recognizable)

This works only if the binary label clearly means 0=Normal, 1=Attack, or uses Normal/Attack text; and the category label clearly marks normal with a conventional name. Otherwise it reports uncertainty instead of guessing.

In [ ]:
def binary_target(s):
    txt = s.astype("string").str.strip().str.casefold()
    return txt.map({"0":0, "1":1, "normal":0, "benign":0, "attack":1, "malicious":1})

label_checked = 0
label_mismatch = 0
label_unknown = 0
if binary_col and category_col:
    for part in pd.read_csv(DATA, chunksize=CHUNK_SIZE, dtype="string", low_memory=False):
        y = binary_target(part[binary_col])
        kind = part[category_col].astype("string").str.strip().str.casefold()
        known_kind = kind.notna() & kind.ne("")
        is_normal = kind.isin(["normal", "benign"])
        compare = y.notna() & known_kind
        label_checked += int(compare.sum())
        label_mismatch += int(((y == 0) != is_normal).fillna(False)[compare].sum())
        label_unknown += int((~compare).sum())
    print("Compared rows:", label_checked)
    print("Binary/category mismatches:", label_mismatch)
    print("Uncheckable rows:", label_unknown)
else:
    print("Automatic label consistency check skipped: binary or category candidate not identified.")


## 6. Save reports and a *draft* feature dictionary

These are descriptive outputs, not final feature selection. Physical meaning, units, scope (network/host/sensor), timing and live-gateway feasibility still need source verification.

In [ ]:
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
DOC_PATH.parent.mkdir(parents=True, exist_ok=True)
summary = {
    "dataset":'ToN-IoT Network',
    "filename":DATA.name,
    "sha256":audit["sha256"],
    "created_utc":datetime.now(timezone.utc).isoformat(),
    "row_count":row_count,
    "candidate_labels":LABEL_COLUMNS,
    "provisional_binary_label":binary_col,
    "provisional_category_label":category_col,
    "exact_duplicate_rows_in_groups":exact_rows_in_duplicate_groups,
    "excess_exact_duplicates":excess_full_duplicates,
    "feature_duplicate_rows_in_groups":feature_rows_in_duplicate_groups,
    "excess_feature_duplicates":excess_feature_duplicates,
    "feature_fingerprint_label_conflict_groups":label_conflicts,
    "binary_category_rows_checked":label_checked,
    "binary_category_mismatches":label_mismatch,
    "binary_category_uncheckable":label_unknown,
    "feature_statistics":quality_rows,
    "caveats":[
        "Hashes are noncryptographic and duplicate counts are screening statistics.",
        "Constant and sparse features must not automatically be dropped before a split decision.",
        "Input values were compared lexically after CSV parsing; raw canonicalization is not established.",
        "Feature meanings/units/time basis are not yet verified from original documentation.",
        "No train/validation/test partitions were created."
    ]
}
with REPORT_PATH.open("w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

lines=[
    f"# ToN-IoT Network — Feature Dictionary DRAFT",
    "",
    f"Source: `ton_iot_train_test_network.csv`; SHA-256: `{audit['sha256']}`",
    "",
    "**Unverified columns:** meaning, units, source modality, temporal window, and deployability must be documented manually.",
    "",
    "| Feature | Numeric parse % | Numeric zeros % | Distinct >= | Missing | Placeholders | Candidate role |",
    "|---|---:|---:|---:|---:|---:|---|",
]
for q in quality_rows:
    name = str(q['feature']).replace('|', chr(92)+'|')
    role = "REVIEW: identifier/time/payload" if q['review_as_identifier_or_payload'] else "VERIFY modality/units"
    lines.append(f"| {name} | {q['numeric_parse_pct']} | {q['numeric_zero_pct_of_all_rows']} | {q['distinct_count_at_least']}{'+' if q['distinct_capped'] else ''} | {q['missing_count']} | {q['placeholder_token_count']} | {role} |")
DOC_PATH.write_text("\n".join(lines)+"\n", encoding="utf-8")
print("Saved feature-quality report:",REPORT_PATH)
print("Saved draft feature dictionary:",DOC_PATH)


## What to share

Send the **candidate label columns and counts** (Step 01), then **exact/feature duplicate counts, feature-label conflict counts, constant fields, most zero-heavy fields, and binary-vs-category consistency check** (Step 02). Then we can compare the three audits before any 70/15/15 split.